In [0]:
import pyspark.sql.functions as F

##1. Access Logs Cleaning

In [0]:
log_regex = r'^(\S+) \S+ \S+ \[([\w:/]+\s[+\-]\d{4})\] "(\S+)\s+(\S+)\s*(\S*)" (\d{3}) (\d+|-)'

In [0]:
df_logs_bronze = spark.read.table("bronze.raw_access_logs")

In [0]:
df_logs_bronze.display()

In [0]:
df_parsed = df_logs_bronze.select(
    F.regexp_extract('value', log_regex, 1).alias('ip'),
    F.regexp_extract('value', log_regex, 2).alias('raw_timestamp'),
    F.regexp_extract('value', log_regex, 3).alias('method'),
    F.regexp_extract('value', log_regex, 4).alias('endpoint'),
    F.regexp_extract('value', log_regex, 5).alias('protocol'),
    F.regexp_extract('value', log_regex, 6).cast('integer').alias('status_code'),
    F.regexp_extract('value', log_regex, 7).cast('integer').alias('content_size')
)

In [0]:
df_logs_silver = df_parsed.withColumn(
    "timestamp", F.to_timestamp("raw_timestamp", "dd/MMM/yyyy:HH:mm:ss Z")
).withColumn(
    "date", F.to_date("timestamp")
).drop("raw_timestamp").filter(F.col("ip") != "")

In [0]:
SILVER_LOGS_TABLE = "silver.access_logs"

df_logs_silver.write.format("delta") \
    .mode("overwrite") \
    .partitionBy("date") \
    .saveAsTable(SILVER_LOGS_TABLE)

print(f"Success! 3.3GB Logs processed and saved to: {SILVER_LOGS_TABLE}")

display(spark.read.table(SILVER_LOGS_TABLE).limit(10))

## 2.Client Hostname Cleaning

In [0]:
df_clients_bronze = spark.read.table("bronze.client_lookup")

In [0]:
df_clients_bronze.display()

In [0]:
df_clients_silver = df_clients_bronze.select(
    F.trim(F.col("client")).alias("ip"),
    F.trim(F.col("hostname")).alias("hostname")
).dropDuplicates(["ip"])


In [0]:
SILVER_CLIENT_TABLE = "silver.client_lookup"

In [0]:
df_clients_silver.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(SILVER_CLIENT_TABLE)

print(f"Success! Cleaned Client Lookup saved to: {SILVER_CLIENT_TABLE}")

display(spark.read.table(SILVER_CLIENT_TABLE).limit(10))